# 10 -- Excited States & TD-DFT: UV/Vis Spectra and Charge-Transfer States

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/10_excited_states_tddft.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the linear-response TDDFT formalism and the Casida equation
- Run TD-DFT excited-state calculations with PySCF
- Compute and plot UV/Vis absorption spectra with Gaussian broadening
- Interpret oscillator strengths and excitation characters
- Recognize the failure of standard functionals for charge-transfer states
- Choose appropriate range-separated functionals (CAM-B3LYP, wB97X-D)

## 1. Theory: Linear-Response TD-DFT

### 1.1 The Casida Equation

Excitation energies are obtained from the generalized eigenvalue problem:

$$\begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{B}^* & \mathbf{A}^* \end{pmatrix}\begin{pmatrix} \mathbf{X} \\ \mathbf{Y} \end{pmatrix} = \omega_I\begin{pmatrix} \mathbf{1} & \mathbf{0} \\ \mathbf{0} & -\mathbf{1} \end{pmatrix}\begin{pmatrix} \mathbf{X} \\ \mathbf{Y} \end{pmatrix}$$

Matrix elements: $A_{ia,jb} = \delta_{ij}\delta_{ab}(\epsilon_a - \epsilon_i) + (ia|jb) + (ia|f_{xc}|jb)$

The **Tamm-Dancoff Approximation (TDA)** sets $\mathbf{B}=0$.

### 1.2 Oscillator Strength

$$f_I = \frac{2}{3}\,\omega_I \sum_\alpha |\langle 0 | \hat{\mu}_\alpha | I \rangle|^2$$

### 1.3 Simulated UV/Vis Spectrum

$$\varepsilon(\omega) = \sum_I f_I \exp\!\left[-\frac{(\omega - \omega_I)^2}{2\sigma^2}\right]$$

In [ ]:
%%time
# =============================================================================
# Ch121a: Quantum Chemistry & DFT -- Notebook 10: Excited States & TD-DFT
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import numpy as np
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
from pyscf import gto, dft

# ------------------------------------------------------------------
# TD-DFT on Formaldehyde (H2C=O): First 8 Excited States
# ------------------------------------------------------------------
mol = gto.M(
    atom='''
        C  0.000  0.000  0.000
        O  0.000  0.000  1.208
        H  0.935  0.000 -0.540
        H -0.935  0.000 -0.540''',
    basis='def2-SVP', charge=0, spin=0, verbose=0
)

mf = dft.RKS(mol)
mf.xc = 'b3lyp'
mf.kernel()

td = mf.TDDFT()
td.nstates = 8
td.kernel()

energies_eV = td.e * 27.2114
osc_str = td.oscillator_strength()

print('Formaldehyde B3LYP/def2-SVP -- TD-DFT Excitation Energies')
print(f'{"State":>6}  {"E (eV)":>8}  {"lam (nm)":>9}  {"f (osc.)"}:>10}')
print('-' * 42)
for i, (e_ev, f) in enumerate(zip(energies_eV, osc_str)):
    lam = 1239.84 / e_ev
    print(f'  S{i+1:>2}    {e_ev:8.3f}   {lam:9.1f}   {f:10.4f}')
print('\nExperimental n->pi* for H2CO: ~4.07 eV (305 nm)')

In [ ]:
# ------------------------------------------------------------------
# Simulated UV/Vis Absorption Spectrum
# ------------------------------------------------------------------
omega = np.linspace(2.0, 12.0, 1000)   # eV axis
sigma = 0.20                             # Gaussian broadening, eV

spectrum = np.zeros_like(omega)
for e, f in zip(energies_eV, osc_str):
    spectrum += f * np.exp(-((omega - e)**2) / (2 * sigma**2))

lam_nm = 1239.84 / omega

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(omega, spectrum, color='steelblue', linewidth=2, label='Broadened')
ax1.vlines(energies_eV, 0, osc_str * 1.5, colors='coral',
           linewidth=1.5, alpha=0.8, label='Stick spectrum')
ax1.set_xlabel('Excitation Energy (eV)', fontsize=12)
ax1.set_ylabel('Intensity (arb. units)', fontsize=12)
ax1.set_title('Formaldehyde: TD-B3LYP/def2-SVP (eV)', fontsize=11)
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

mask = (lam_nm > 100) & (lam_nm < 700)
ax2.plot(lam_nm[mask], spectrum[mask], color='purple', linewidth=2)
ax2.set_xlabel('Wavelength (nm)', fontsize=12)
ax2.set_ylabel('Intensity (arb. units)', fontsize=12)
ax2.set_title('Formaldehyde: TD-B3LYP/def2-SVP (nm)', fontsize=11)
ax2.invert_xaxis(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
%%time
# ------------------------------------------------------------------
# Functional / Basis Comparison for the Lowest n->pi* Transition
# ------------------------------------------------------------------
import pandas as pd
from pyscf import gto, dft

def tddft_lowest(atom_str, xc, basis):
    m = gto.M(atom=atom_str, basis=basis, charge=0, spin=0, verbose=0)
    mf2 = dft.RKS(m); mf2.xc = xc; mf2.kernel()
    td2 = mf2.TDDFT(); td2.nstates = 3; td2.kernel()
    e1_ev = td2.e[0] * 27.2114
    f1 = td2.oscillator_strength()[0]
    return round(e1_ev, 3), round(f1, 4)

h2co = 'C 0 0 0; O 0 0 1.208; H 0.935 0 -0.540; H -0.935 0 -0.540'

rows = []
for xc, basis in [('b3lyp','def2-SVP'), ('b3lyp','def2-TZVP'), ('cam-b3lyp','def2-SVP')]:
    e1, f1 = tddft_lowest(h2co, xc, basis)
    rows.append({'Functional': xc, 'Basis': basis, 'E1 (eV)': e1, 'f1': f1})

df = pd.DataFrame(rows)
print('Lowest excitation of formaldehyde -- method comparison')
print(df.to_string(index=False))
print('\nExperimental n->pi*: 4.07 eV (305 nm)')

## 2. Charge-Transfer States & Range-Separated Functionals

Standard hybrid functionals (B3LYP, PBE0) **underestimate** charge-transfer (CT)
excitation energies due to self-interaction error and incorrect long-range Coulomb asymptotics.

**Solution -- range-separated hybrids** separate exchange into short-range (DFT)
and long-range (exact HF) using the error function:

$$\frac{1}{r_{12}} = \frac{1-\text{erf}(\omega r_{12})}{r_{12}} + \frac{\text{erf}(\omega r_{12})}{r_{12}}$$

| Functional | SR exchange | LR exchange | Recommended for |
|------------|:-----------:|:----------:|:----------------|
| B3LYP      | 20% global  | 20% global | Local excitations |
| CAM-B3LYP  | 19%         | 65%        | CT states, Rydberg |
| wB97X-D    | ~20%        | 100%       | General; includes dispersion |

## Research Connection

TD-DFT is the workhorse method for UV/Vis spectroscopy in chemistry and biochemistry:

- **Photochemistry**: S1 and T1 state energies guide design of photocatalysts and photoredox reagents.
- **Dye-sensitised solar cells**: TD-DFT predicts absorption maxima of organic sensitisers.
- **Bioimaging**: Fluorescent protein chromophores (GFP, mCherry) are screened with TD-DFT.
- **OLEDs**: Singlet/triplet gaps in Ir and Pt phosphors require TD-DFT with CAM-B3LYP.

## Summary

| Concept | Key Formula / Fact |
|---------|-------------------|
| Casida equation | $(\mathbf{A}+\mathbf{B})(\mathbf{A}-\mathbf{B})\mathbf{F}=\omega^2\mathbf{F}$ |
| Oscillator strength | $f_I = \frac{2}{3}\omega_I|\langle 0|\hat{\mu}|I\rangle|^2$ |
| TDA | Set $\mathbf{B}=0$; cheaper, usually accurate |
| CT state problem | Standard hybrids underestimate by 0.5--2 eV |
| CAM-B3LYP | Range-separated; correct $-1/r$ asymptotics |
| Stokes shift | $\Delta E = E_{\rm abs} - E_{\rm em}$; due to geometry relaxation in S1 |

## Exercises

1. **Ethylene TD-DFT**: Compute the first 4 excited states of ethylene (C2H4) at
   B3LYP/def2-SVP. Identify the pi->pi* transition. Compare with experiment at ~165 nm.

2. **Functional comparison**: For formaldehyde, compare the n->pi* excitation from TDHF,
   TD-B3LYP, and TD-CAM-B3LYP. Which is closest to experiment (4.07 eV)?

3. **Oscillator strengths**: Among the first 8 states of formaldehyde, which is the
   brightest (largest oscillator strength)?

4. **Broadening**: Re-plot the UV/Vis spectrum with sigma = 0.05, 0.20, and 0.50 eV.
   How does broadening affect band resolution?

5. **TDA vs full TDDFT**: Run TDA (`td = mf.TDA()`) vs full TDDFT on formaldehyde.
   Compare the lowest excitation energies. When does TDA break down?